In [1]:
# Zelle 1: Bibliotheken laden. Alle weiteren Zellen benutzen diese Namen (pn, plf, np, pd, nx, plt).
import pypowsybl as pp  # Netzmodell und Lastfluss (pypowsybl)
import pypowsybl.network as pn  # Netze laden und ändern: pn.load(...), update_lines(...), get_buses(...)
import pypowsybl.loadflow as plf  # Lastflussrechnung: plf.run_ac(...), plf.run_dc(...)
import numpy as np  # Matrizen und Vektoren
from pypowsybl_jupyter import NetworkMapWidget, network_explorer  # interaktive Netzansicht im Notebook
import matplotlib.pyplot as plt  # Diagramme (Zelle 5)
import pandas as pd  # Tabellen
import math as m
import os
import networkx as nx  # Graphen: Inseln finden (Zelle 5 und 11)
import pandas as pd


In [2]:
# Zelle 2: KONFIGURATION. Nur hier muss etwas eingetragen werden, alle weiteren Zellen laufen dann einfach durch.
#
# NETZ = das Netz, das geprüft werden soll. Zwei Möglichkeiten:
#   a) Pfad zu einer Netzdatei (xiidm, uct, ...):  NETZ = "/home/.../mein_netz.xiidm"
#   b) pypowsybl-Testnetz (ohne Klammern!):        NETZ = pn.create_ieee14   (auch create_ieee30, create_ieee57, ...)
from pathlib import Path

NETZ = pn.create_ieee14
#NETZ = "/home/vc8596/ToOp/data/complex_grid/grid.xiidm"


# Es werden alle Ausfallmengen mit 1 bis MAX_AUSFAELLE Zweigen geprüft, die das Netz in mehr Inseln teilen.
# Die Rechenzeit wächst stark mit diesem Wert. Richtwert: complex_grid (30 Zweige) mit 4 dauert etwa 2 Minuten.
MAX_AUSFAELLE = 3

# Schutz gegen zu große Läufe: Wenn es mehr Kombinationen gäbe, bricht die Rechnung mit einer Meldung ab.
MAX_KOMBINATIONEN = 200_000

# Abweichungen unterhalb dieser Schwelle (in MW) gelten als übereinstimmend
TOL = 1e-6

# Für Zelle 6 (Tabelle Zweig für Zweig): Welche Ausfallmenge soll angesehen werden?
#   None = der Fall mit der größten Abweichung (im unten gewählten Slack-Modell)
#   oder eine Liste von Zweig-IDs, z.B. ["L2-3-1", "L3-4-1"] (die IDs stehen in Zelle 5 in der Spalte outaged_branches)
AUSFALL_ANSEHEN = None
# Mit welchem Slack-Modell? "einzelner Slack" oder "verteilter Slack"
SLACK_ANSEHEN = "einzelner Slack"

In [ ]:
# Zelle 3: Hilfsfunktionen. Hier muss nichts geändert werden.
# Idee: Für jede Ausfallmenge, die das Netz in mehr Inseln teilt, wird compute_split_modf_main_grid mit einem
# pypowsybl-DC-Lastfluss auf dem wirklich aufgetrennten Netz verglichen.
import itertools
from collections.abc import Callable
from math import comb

import jax
import jax.numpy as jnp
from toop_engine_dc_solver.jax.split_modf_main_grid import compute_split_modf_main_grid  # die zu testende Funktion

jax.config.update("jax_enable_x64", True)  # doppelte Genauigkeit, sonst stimmt der Vergleich mit pypowsybl nicht

BRANCH_COLUMNS = ["bus1_id", "bus2_id", "connected1", "connected2"]
# Slack-Modell 1: ein einziger Slack-Bus gleicht die Bilanz aus (so rechnet auch compute_split_modf_main_grid)
SINGLE_SLACK = plf.Parameters(distributed_slack=False)
# Slack-Modell 2: der Leistungsfehler wird anteilig auf alle Generatoren verteilt (Standard im Produktions-Runner)
DISTRIBUTED_SLACK = plf.Parameters(distributed_slack=True, balance_type=plf.BalanceType.PROPORTIONAL_TO_GENERATION_P)


def make_factory(netz) -> Callable[[], pn.Network]:
    """Macht aus NETZ (Pfad oder pypowsybl-Funktion) eine Funktion, die bei jedem Aufruf ein frisches Netz liefert."""
    if callable(netz):
        return netz
    path = Path(netz)
    return lambda: pn.load(path)


# Wirkleistung am Anfang aller Leitungen und Trafos als eine Liste (Index = Zweig-ID)
def _branch_flows(net: pn.Network) -> pd.Series:
    return pd.concat([net.get_lines(attributes=["p1"]), net.get_2_windings_transformers(attributes=["p1"])]).p1


# DC-Lastfluss rechnen, auch wenn das Netz in mehrere Inseln zerfällt (ALL_CONNECTED = alle Inseln rechnen)
def _run_dc(net: pn.Network, params: plf.Parameters) -> None:
    all_components = plf.Parameters(
        distributed_slack=params.distributed_slack,
        balance_type=params.balance_type,
        component_mode=plf.ComponentMode.ALL_CONNECTED,
    )
    plf.run_dc(net, all_components)


def _dc_state(net, params, branches, incidence):
    """DC-Lastfluss rechnen und zurückgeben: Zweigflüsse und Winkeldifferenz über jedem Zweig."""
    _run_dc(net, params)
    angles = np.deg2rad(net.get_buses().v_angle.loc[net.get_buses().index].to_numpy())
    return _branch_flows(net).loc[branches.index].to_numpy(), incidence @ angles


def _build_model(factory: Callable[[], pn.Network], params: plf.Parameters) -> dict:
    """Baut aus dem intakten Netz alles, was compute_split_modf_main_grid braucht (PTDF, Basisflüsse, Einspeisungen)."""
    net = factory()
    # Drei-Wicklungs-Trafos sind in dieser Auswertung nicht enthalten (ihr Fluss würde in der Bilanz fehlen)
    if len(net.get_3_windings_transformers()) > 0:
        raise ValueError(
            "Das Netz enthält 3-Wicklungs-Trafos. Diese werden hier nicht unterstützt (als 2-Wicklungs-Trafos modellieren)."
        )
    # Alle Zweige sammeln, die an beiden Enden verbunden sind
    branches = pd.concat(
        [net.get_lines(attributes=BRANCH_COLUMNS), net.get_2_windings_transformers(attributes=BRANCH_COLUMNS)]
    )
    branches = branches[branches.connected1 & branches.connected2]
    # Busindex je Zweigende und Zweig-Knoten-Inzidenzmatrix (+1 am Anfangsbus, -1 am Endbus)
    buses = net.get_buses().index
    from_node = buses.get_indexer(branches.bus1_id)
    to_node = buses.get_indexer(branches.bus2_id)
    incidence = np.zeros((len(branches), len(buses)))
    incidence[np.arange(len(branches)), from_node] = 1.0
    incidence[np.arange(len(branches)), to_node] = -1.0

    # Ein erster DC-Lauf liefert den Slack-Bus von pypowsybl und die Referenzwerte für die Suszeptanz-Schätzung
    results = plf.run_dc(net, SINGLE_SLACK)
    slack = buses.get_loc(results[0].reference_bus_id)
    fit_flow, fit_angle = _dc_state(net, SINGLE_SLACK, branches, incidence)
    # Basisflüsse n_0_flow mit dem gewählten Slack-Modell (params). Sie sind der Normalzustand vor dem Ausfall.
    _run_dc(net, params)
    n_0_flow = _branch_flows(net).loc[branches.index].to_numpy()

    # Suszeptanzen schätzen: 6 DC-Läufe mit zufällig veränderten Einspeisungen (-50 % bis +50 %).
    # Flussänderung = b * Winkeldifferenz-Änderung; Phasenschieber-Versätze heben sich in den Differenzen auf.
    rng = np.random.default_rng(0)
    d_flow, d_angle = [], []
    for _ in range(6):
        perturbed = factory()
        gens, loads = perturbed.get_generators(), perturbed.get_loads()
        perturbed.update_generators(
            id=gens.index.tolist(), target_p=(gens.target_p * rng.uniform(0.5, 1.5, len(gens))).tolist()
        )
        perturbed.update_loads(id=loads.index.tolist(), p0=(loads.p0 * rng.uniform(0.5, 1.5, len(loads))).tolist())
        flow, angle = _dc_state(perturbed, SINGLE_SLACK, branches, incidence)
        d_flow.append(flow - fit_flow)
        d_angle.append(angle - fit_angle)
    d_flow, d_angle = np.array(d_flow), np.array(d_angle)
    susceptance = (d_flow * d_angle).sum(0) / (d_angle**2).sum(0)
    # Kontrolle: Passt "Flussänderung = b * Winkeländerung" auf allen Zweigen? Sonst wäre die PTDF falsch.
    if not np.all(np.isfinite(susceptance)) or np.abs(d_flow - susceptance * d_angle).max() > 1e-6:
        raise ValueError("Die Suszeptanzen lassen sich nicht aus den DC-Läufen schätzen (z.B. Zweig ohne Winkeldifferenz).")

    # PTDF = diag(b) * A * B_red^-1; die Spalte des Slack-Busses bleibt 0 (Winkelreferenz)
    keep = np.arange(len(buses)) != slack
    bus_matrix = incidence.T @ (susceptance[:, None] * incidence)
    ptdf = np.zeros_like(incidence)
    ptdf[:, keep] = (susceptance[:, None] * incidence[:, keep]) @ np.linalg.inv(bus_matrix[np.ix_(keep, keep)])
    # Rückgabe: Zweige, Anzahl Busse, Zweigenden, Slack, Basisflüsse, Einspeisung je Bus (= A^T * Flüsse) und PTDF
    return {
        "branches": branches,
        "n_buses": len(buses),
        "from_node": from_node,
        "to_node": to_node,
        "slack": slack,
        "n_0_flow": n_0_flow,
        "nodal_injection": incidence.T @ n_0_flow,
        "ptdf": ptdf,
    }


def _island_creating_outages(model: dict, max_outages: int) -> list[np.ndarray]:
    """Findet alle Ausfallmengen mit bis zu max_outages Zweigen, die die Zahl der elektrischen Inseln erhöhen."""
    edges = list(zip(model["from_node"].tolist(), model["to_node"].tolist(), strict=True))
    n_combinations = sum(comb(len(edges), k) for k in range(1, max_outages + 1))
    if n_combinations > MAX_KOMBINATIONEN:
        raise ValueError(
            f"{n_combinations} Kombinationen sind mehr als MAX_KOMBINATIONEN={MAX_KOMBINATIONEN}. "
            "MAX_AUSFAELLE verkleinern oder MAX_KOMBINATIONEN erhöhen."
        )

    # Zählt die Inseln (zusammenhängende Teilnetze), wenn die Zweige in 'removed' fehlen
    def n_components(removed: set[int]) -> int:
        graph = nx.Graph()
        graph.add_nodes_from(range(model["n_buses"]))
        graph.add_edges_from(edge for index, edge in enumerate(edges) if index not in removed)
        return nx.number_connected_components(graph)

    # Alle Kombinationen aus 1 bis max_outages Zweigen probieren; behalten werden nur die, die mehr Inseln erzeugen
    intact = n_components(set())
    return [
        np.array(outage_set)
        for n_outages in range(1, max_outages + 1)
        for outage_set in itertools.combinations(range(len(edges)), n_outages)
        if n_components(set(outage_set)) > intact
    ]


# Schaltet die angegebenen Zweige ab (beide Seiten), egal ob Leitung oder Trafo
def _open_branches(net: pn.Network, branch_ids: list[str]) -> None:
    lines = net.get_lines().index
    for branch_id in branch_ids:
        update = net.update_lines if branch_id in lines else net.update_2_windings_transformers
        update(id=branch_id, connected1=False, connected2=False)


def compare_with_powsybl(factory: Callable[[], pn.Network], params: plf.Parameters, max_outages: int) -> pd.DataFrame:
    """Vergleicht split MODF mit pypowsybl DC auf der behaltenen Insel für jede inselbildende Ausfallmenge.

    Liefert eine Zeile je Ausfallmenge. Zusätzlich steht in discarded_imbalance_mw die Leistungsbilanz der
    abgetrennten Inseln, die compute_split_modf_main_grid verwirft und am Slack-Bus ausgleicht.
    """
    model = _build_model(factory, params)
    branches = model["branches"]
    outage_sets = _island_creating_outages(model, max_outages)
    if not outage_sets:
        raise ValueError("Keine Ausfallmenge teilt das Netz in mehr Inseln (z.B. ein vollständig vermaschtes Netz).")

    # Ausfallmengen unterschiedlicher Größe werden mit -1 auf die gleiche Länge aufgefüllt (-1 = kein Zweig)
    split_flow, _main_bus, main_monitored, imbalance, _rank_loss = compute_split_modf_main_grid(
        ptdf=jnp.asarray(model["ptdf"]),
        from_node=jnp.asarray(model["from_node"], dtype=jnp.int32),
        to_node=jnp.asarray(model["to_node"], dtype=jnp.int32),
        nodal_injections=jnp.asarray(model["nodal_injection"][None, :]),
        n_0_flow=jnp.asarray(model["n_0_flow"][None, :]),
        outages=jnp.asarray(
            np.stack([np.pad(s, (0, max_outages - len(s)), constant_values=-1) for s in outage_sets]), dtype=jnp.int32
        ),
        branches_monitored=jnp.arange(len(branches), dtype=jnp.int32),
        slack_bus=jnp.asarray(model["slack"], dtype=jnp.int32),
    )
    # split_flow[0] = Flüsse (Zeitschritt 0); NaN bei Zweigen außerhalb der behaltenen Insel.
    # main_monitored = Maske der Zweige, die in der behaltenen Insel liegen.
    split_flow = np.asarray(split_flow[0])
    main_monitored = np.asarray(main_monitored)
    discarded_imbalance = np.abs(np.asarray(imbalance)[0])

    # Je Ausfallmenge: frisches Netz laden, Zweige öffnen, DC rechnen, Fluss als Referenz nehmen
    rows = []
    for case, outage_set in enumerate(outage_sets):
        outaged_ids = branches.index[outage_set].tolist()
        cut_net = factory()
        _open_branches(cut_net, outaged_ids)
        _run_dc(cut_net, params)
        reference = _branch_flows(cut_net).reindex(branches.index).to_numpy()
        # Ausgefallene Zweige haben Fluss 0 (pypowsybl liefert dort NaN)
        reference[outage_set] = 0.0
        # Abweichung nur auf Zweigen der behaltenen Insel, dort muss split MODF mit pypowsybl übereinstimmen
        difference = np.abs(split_flow[case] - reference)[main_monitored[case]]
        max_diff = np.max(difference, initial=0.0)
        rows.append(
            {
                "outaged_branches": ", ".join(outaged_ids),
                "n_outages": len(outaged_ids),
                "n_kept_branches": int(main_monitored[case].sum()),
                "max_diff_mw": max_diff,
                "matches": bool(max_diff <= TOL),
                "discarded_imbalance_mw": discarded_imbalance[case],
            }
        )
    return pd.DataFrame(rows)

In [ ]:
# Zelle 4: Prüfung ausführen. Hier muss nichts geändert werden (Netz und Größe stehen in Zelle 2).
# Es werden beide Slack-Modelle gerechnet: einzelner Slack (so rechnet die Funktion) und verteilter Slack.
factory = make_factory(NETZ)
slack_models = {"einzelner Slack": SINGLE_SLACK, "verteilter Slack": DISTRIBUTED_SLACK}
comparisons = {name: compare_with_powsybl(factory, params, MAX_AUSFAELLE) for name, params in slack_models.items()}

# Zusammenfassung: eine Zeile je Slack-Modell
# - Ausfallmengen: wie viele Kombinationen das Netz in mehr Inseln teilen (nur diese werden geprüft)
# - abweichend: bei wie vielen davon die Funktion von pypowsybl abweicht (mehr als TOL)
# - größte Abweichung: in MW, nur auf den Zweigen der Insel mit dem Slack-Bus
pd.DataFrame(
    [
        {
            "Slack-Modell": name,
            "Ausfallmengen": len(df),
            "abweichend": int((~df.matches).sum()),
            "größte Abweichung [MW]": df.max_diff_mw.max(),
        }
        for name, df in comparisons.items()
    ]
).set_index("Slack-Modell")

,Ausfallmengen,abweichend,größte Abweichung [MW]
Slack-Modell,,,
einzelner Slack,345,0,1.307399e-12
verteilter Slack,345,124,1.383260e+01


In [ ]:
# Zelle 5: Details zu den Abweichungen beim verteilten Slack.
# Ursache: compute_split_modf_main_grid wirft die Einspeisungen der abgetrennten Inseln weg und legt die übrige
# Bilanz (discarded_imbalance_mw) auf den ursprünglichen Slack-Bus. pypowsybl verteilt sie dagegen auf die Generatoren.
# Kreuztabelle: Zeilen = wurde eine Bilanz verworfen?, Spalten = stimmt das Ergebnis mit pypowsybl überein?
distributed = comparisons[SLACK_ANSEHEN]
display(pd.crosstab(distributed.discarded_imbalance_mw > 1e-6, distributed.matches, rownames=["verworfene Bilanz"]))
# Die 10 größten Abweichungen beim verteilten Slack
distributed.sort_values("max_diff_mw", ascending=False).head(10)

matches,True
verworfene Bilanz,
False,202
True,143


,outaged_branches,n_outages,n_kept_branches,max_diff_mw,matches,discarded_imbalance_mw
65,"L1-2-1, L7-8-1, T4-9-1",3,19,1.307399e-12,True,1.776357e-14
64,"L1-2-1, L7-8-1, T4-7-1",3,19,9.663381e-13,True,1.776357e-14
61,"L1-2-1, L7-8-1, L10-11-1",3,19,9.094947e-13,True,1.776357e-14
59,"L1-2-1, L7-8-1, L9-10-1",3,19,9.094947e-13,True,1.776357e-14
68,"L1-2-1, L9-10-1, L10-11-1",3,18,8.242296e-13,True,9.000000e+00
67,"L1-2-1, L7-9-1, T4-7-1",3,17,7.958079e-13,True,2.486900e-14
53,"L1-2-1, L6-11-1, L9-10-1",3,17,7.958079e-13,True,1.250000e+01
52,"L1-2-1, L6-11-1, L7-8-1",3,19,7.673862e-13,True,1.776357e-14
47,"L1-2-1, L2-3-1, L7-8-1",3,19,7.389644e-13,True,1.776357e-14
63,"L1-2-1, L7-8-1, L13-14-1",3,19,7.389644e-13,True,1.776357e-14


In [ ]:
# Zelle 6: Tabelle Zweig für Zweig für EINE Ausfallmenge (Einstellung AUSFALL_ANSEHEN und SLACK_ANSEHEN in Zelle 2).
# Jede Zeile ist ein Zweig (Leitung oder Trafo). Die Spalten stellen die Flüsse gegenüber:
#   vor Ausfall   = Fluss im intakten Netz
#   Split MODF    = Fluss nach dem Ausfall laut compute_split_modf_main_grid (NaN = Zweig nicht in der behaltenen Insel)
#   pypowsybl     = Fluss nach dem Ausfall laut pypowsybl-DC-Lastfluss auf dem wirklich aufgetrennten Netz
#   Differenz     = Split MODF minus pypowsybl
slack_params = slack_models[SLACK_ANSEHEN]
if AUSFALL_ANSEHEN is None:
    # Fall mit der größten Abweichung aus Zelle 4 nehmen
    worst_case = comparisons[SLACK_ANSEHEN].sort_values("max_diff_mw", ascending=False).iloc[0]
    ausfall_ids = worst_case.outaged_branches.split(", ")
else:
    ausfall_ids = list(AUSFALL_ANSEHEN)
print("Ausfallmenge:", ausfall_ids, "| Slack-Modell:", SLACK_ANSEHEN)

# Modell des intakten Netzes (Basisflüsse, PTDF, Einspeisungen) und Index der ausgefallenen Zweige
view_model = _build_model(factory, slack_params)
view_branches = view_model["branches"]
view_outage = view_branches.index.get_indexer(ausfall_ids)
if (view_outage < 0).any():
    raise ValueError(f"Unbekannte Zweig-ID in {ausfall_ids}. Erlaubt sind die Zweige aus view_branches.index.")

# Split MODF für diese eine Ausfallmenge rechnen
view_flow, _, view_main_monitored, view_imbalance, _ = compute_split_modf_main_grid(
    ptdf=jnp.asarray(view_model["ptdf"]),
    from_node=jnp.asarray(view_model["from_node"], dtype=jnp.int32),
    to_node=jnp.asarray(view_model["to_node"], dtype=jnp.int32),
    nodal_injections=jnp.asarray(view_model["nodal_injection"][None, :]),
    n_0_flow=jnp.asarray(view_model["n_0_flow"][None, :]),
    outages=jnp.asarray(view_outage[None, :], dtype=jnp.int32),
    branches_monitored=jnp.arange(len(view_branches), dtype=jnp.int32),
    slack_bus=jnp.asarray(view_model["slack"], dtype=jnp.int32),
)

# Referenz: pypowsybl-DC-Lastfluss auf dem Netz mit geöffneten Zweigen (ausgefallene Zweige = 0)
view_cut_net = factory()
_open_branches(view_cut_net, ausfall_ids)
_run_dc(view_cut_net, slack_params)
view_reference = _branch_flows(view_cut_net).reindex(view_branches.index).to_numpy()
view_reference[view_outage] = 0.0

print(f"Verworfene Leistungsbilanz der abgetrennten Inseln: {float(view_imbalance[0, 0]):.2f} MW")
view_table = pd.DataFrame(
    {
        "von_bus": view_branches.bus1_id,
        "nach_bus": view_branches.bus2_id,
        "ausgefallen": view_branches.index.isin(ausfall_ids),
        "in behaltener Insel": np.asarray(view_main_monitored[0]),
        "vor Ausfall [MW]": view_model["n_0_flow"],
        "Split MODF [MW]": np.asarray(view_flow[0, 0]),
        "pypowsybl [MW]": view_reference,
    },
    index=view_branches.index,
)
view_table["Differenz [MW]"] = view_table["Split MODF [MW]"] - view_table["pypowsybl [MW]"]
view_table.round(3)

Ausfallmenge: ['L1-2-1', 'L7-8-1', 'T4-9-1'] | Slack-Modell: einzelner Slack
Verworfene Leistungsbilanz der abgetrennten Inseln: -0.00 MW


,von_bus,nach_bus,ausgefallen,in behaltener Insel,vor Ausfall [MW],Split MODF [MW],pypowsybl [MW],Differenz [MW]
id,,,,,,,,
L1-2-1,VL1_0,VL2_0,True,True,147.839,0.000,0.000,0.0
L1-5-1,VL1_0,VL5_0,False,True,71.161,219.000,219.000,0.0
L2-3-1,VL2_0,VL3_0,False,True,70.015,44.833,44.833,-0.0
L2-4-1,VL2_0,VL4_0,False,True,55.152,2.453,2.453,-0.0
L2-5-1,VL2_0,VL5_0,False,True,40.972,-28.986,-28.986,-0.0
L3-4-1,VL3_0,VL4_0,False,True,-24.185,-49.367,-49.367,0.0
L4-5-1,VL4_0,VL5_0,False,True,-61.746,-129.960,-129.960,-0.0
L6-11-1,VL6_0,VL11_0,False,True,6.728,12.549,12.549,0.0
L6-12-1,VL6_0,VL12_0,False,True,7.607,8.462,8.462,0.0
